# 2.1.3 Improved LSTM run

## 1. Business hypothesis and experiment overview

**Owner:** Winston Ho Rui Wei  
**Status:** completed run `20261104_175920_improved_seed42`; executed outputs and findings are preserved below.

The ablation study found that explicit freshness substantially improved retrieval.
This experiment fits one LSTM with target-aware attention and scores articles as
`neural_logit - freshness_weight * log1p(age_hours)`. It reuses the neural
hyperparameters selected by the initial Optuna study, preserving a controlled
comparison. It does not ensemble multiple seeds or add a new network.

At every epoch, a fixed 2,000-request validation sample selects the best freshness
weight from the ablation grid and controls checkpoint selection. Optuna records
every epoch/weight trial. Recall@100 is primary, nDCG@100 breaks ties, earlier
epochs are retained on equal metrics, and smaller weights break within-epoch ties.
The same requests are excluded from the completed ablation reporting sample.

Inference batches requests and candidate tiles, reuses cached representations,
performs exact score/article-ID top-k selection on GPU, and writes columnar
Parquet batches locally before copying completed files to shared Drive. A small
saved-model benchmark checks the new path against the original on Colab before
training. Section 9 reports the executed benchmark, validation findings and limitations.

The full time-eligible catalogue is scored for every request. The working
shortlist is the first 100; 200 are exported as a buffer. A separate business
reranker will produce the final five. Unshown article outcomes remain unknown.

## 2. Setup and run configuration

Select **A100 GPU** and **High RAM** in Colab. Upload `lstm_helpers.zip` to the
runtime, then run the cells in order. This archive contains the three source
helpers kept beside this notebook in the repository; no GitHub login is needed.

The shared Drive shortcut and the completed initial run must be present at the
paths below. The initial model can be read from its original or organised location. New results
go to `Experiments/2.1_lstm/2.1.3_improved_run/<run_id>/`. This notebook follows one path: saved-model inference benchmark,
fresh single-model training with joint epoch/freshness selection, complete
exports, persistent saving, and numerical reload of the final score.

In [1]:
from pathlib import Path

EXPERIMENT_ID = "2.1_lstm"
EXPERIMENT_STAGE = "2.1.3"
OWNER = "Winston Ho Rui Wei"
SEED = 42
FEATURE_CASE = "raw"
DATA_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder")
REFERENCE_RUN_ID = "20261104_014914_seed42"
HISTORY_MAX_LEN = 50
SELECTION_K, EXPORT_K, SHORTLIST_SIZE, FINAL_DISPLAY_SIZE = 100, 200, 100, 5
SELECTION_REQUESTS = 2000
FRESHNESS_GRID = (0., 0.25, 0.5, 1., 2., 4., 8., 16., 64.)
PAIR_CHUNK, REQUEST_BATCH = 131072, 16
RELOAD_TOLERANCE = 1e-4
SELECTION_OBJECTIVE = "known_click_recall@100"
SELECTION_TIEBREAK = "observed_click_ndcg@100"

In [2]:
import os, sys, subprocess, platform, json, shutil, time, hashlib, importlib.util, sqlite3
from importlib import metadata
from dataclasses import fields, asdict

if "google.colab" in sys.modules or Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
    CODE_DIR = Path("/content/lstm_improved_code")
    CODE_DIR.mkdir(exist_ok=True)
    import zipfile
    with zipfile.ZipFile("/content/lstm_helpers.zip") as archive:
        archive.extractall(CODE_DIR)
else:
    CODE_DIR = Path.cwd()
for name in ("lstm_data.py", "lstm_model.py", "lstm_inference.py"):
    if not (CODE_DIR / name).is_file():
        raise FileNotFoundError(f"Place {name} in {CODE_DIR} before continuing.")

# Keep the runtime's core stack; install only the small missing dependencies.
for dist in ("duckdb", "optuna"):
    try:
        metadata.version(dist)
    except metadata.PackageNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", dist], check=True)
sys.path.insert(0, str(CODE_DIR))
for name in ("lstm_data", "lstm_model", "lstm_inference"):
    sys.modules.pop(name, None)
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import optuna
from IPython.display import display
import lstm_data as D
import lstm_model as M
import lstm_inference as F

assert D.EXPORT_SCHEMA_VERSION == 4, "Upload the current schema-v4 helpers."
assert Path(D.__file__).resolve().parent == CODE_DIR.resolve()
assert Path(M.__file__).resolve().parent == CODE_DIR.resolve()
PATHS, PATH_NOTES = D.resolve_paths(DATA_ROOT)
SPLIT_MANIFEST = json.loads((DATA_ROOT / "split_manifest.json").read_text())
if SPLIT_MANIFEST.get("protocol") != "ebnerd_small_labelled_user_holdout_v1":
    raise ValueError("Expected the labelled project dataset and its split manifest.")
if json.loads((DATA_ROOT / "feature_engineering" / "split_manifest.json").read_text()) != SPLIT_MANIFEST:
    raise ValueError("Feature outputs and dataset split manifest do not match.")

RUN_ID = pd.Timestamp.now(tz="Asia/Singapore").strftime("%Y%m%d_%H%M%S") + "_improved_seed42"
LOCAL_ROOT = Path("/content/lstm_outputs") if Path("/content").is_dir() else CODE_DIR / "outputs"
RUN_DIR = LOCAL_ROOT / RUN_ID
PERSIST_RUN_DIR = DATA_ROOT / "Experiments" / EXPERIMENT_ID / "2.1.3_improved_run" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
D.write_json(SPLIT_MANIFEST, RUN_DIR / "split_manifest.json")
PERSIST_RUN_DIR.mkdir(parents=True, exist_ok=False)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
M.set_seed(SEED)
VERSIONS = {dist: metadata.version(dist) for dist in
            ("torch", "numpy", "pandas", "pyarrow", "duckdb", "optuna")}
INPUTS = D.describe_inputs(PATHS, hash_files=False)  # path/size/mtime, not content hashes

def log(*parts):
    print(*parts, flush=True)

def copy_tree(source, destination):
    # Copy completed files atomically. The live Optuna database is outside RUN_DIR.
    for source_file in sorted(Path(source).rglob("*")):
        if source_file.is_file():
            D._copy_atomic(source_file, Path(destination) / source_file.relative_to(source))

log("Run", RUN_ID, "on", DEVICE, "| persistent folder:", PERSIST_RUN_DIR)
log("Dependencies:", VERSIONS)
log("GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU; select an A100 runtime")
REFERENCE_DIR = DATA_ROOT / "Experiments" / EXPERIMENT_ID / "2.1.1_initial_run" / REFERENCE_RUN_ID
if not REFERENCE_DIR.is_dir():
    REFERENCE_DIR = DATA_ROOT / "Experiments" / EXPERIMENT_ID / REFERENCE_RUN_ID
REFERENCE_CONFIG = json.loads((REFERENCE_DIR / "run_config.json").read_text())
REFERENCE_TUNING = json.loads((REFERENCE_DIR / "tuning" / "summary.json").read_text())
MODEL_KEYS = ("article_dim", "category_dim", "type_dim", "topic_dim", "text_dim", "image_dim",
              "lstm_hidden", "lstm_layers", "attention_dim", "interaction_dim", "head_hidden",
              "dropout", "embedding_dropout", "use_id_embedding", "id_dim", "text_dropout")
MODEL_HPARAMS = {k: REFERENCE_CONFIG["model"][k] for k in MODEL_KEYS}
TRAIN_HPARAMS = {k: v for k, v in REFERENCE_CONFIG["train_config"].items() if k != "seed"}
D.write_json({"reference_run_id": REFERENCE_RUN_ID, "neural_tuning": REFERENCE_TUNING,
              "model_hparams": MODEL_HPARAMS, "train_hparams": TRAIN_HPARAMS},
             RUN_DIR / "reference_hyperparameters.json")

Mounted at /content/drive
Run 20261104_175920_improved_seed42 on cuda | persistent folder: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/2.1.3_improved_run/20261104_175920_improved_seed42
Dependencies: {'torch': '2.11.0+cu130', 'numpy': '2.4.6', 'pandas': '2.2.3', 'pyarrow': '23.0.1', 'duckdb': '1.3.2', 'optuna': '5.0.0'}
GPU: NVIDIA A100-SXM4-80GB


## 3. Load and check shared data

Use the existing expanded candidate tables, original request keys and fixed
pre-period histories. Fit the neural loss only on clicked and shown-but-unclicked
articles. The labelled project test is loaded with its outcomes masked; it is
scored for handoff but never used for selection or development evaluation.
All splits use the 20,738-article EB-NeRD SMALL catalogue prepared by Step 1.
Both article files contain the same snapshot, and both embedding folders contain
the complete SMALL article-ID lookup, including missing-vector rows.


In [3]:
FRAMES, OFFSETS, TABLE_INFO, SPLIT_START = {}, {}, {}, {}
for split in D.SPLITS:
    frame, info = D.load_candidate_table(PATHS[f"features_{split}"], split)
    offsets = info.pop("offsets")
    SPLIT_START[split] = pd.Timestamp(frame["impression_time"].min())
    if PATHS.get(f"behaviors_{split}"):
        check = D.cross_check_behaviours(frame, PATHS[f"behaviors_{split}"], split)
        assert check["feature_rows_not_in_behaviours"] == 0, check
        assert check["behaviour_candidates_not_in_features"] == 0, check
        assert check.get("label_mismatches", 0) == 0, check
        assert info["impressions"] == check["behaviour_rows"], "Missing original requests"
        info["behaviour_cross_check"] = check
    FRAMES[split], OFFSETS[split], TABLE_INFO[split] = frame, offsets, info
DEVICE_COLUMNS = TABLE_INFO["train"]["device_columns"]
assert all(TABLE_INFO[s]["device_columns"] == DEVICE_COLUMNS for s in D.SPLITS)
assert TABLE_INFO["test"]["labelled"] is False
log("All recorded requests/candidates:",
    {s: (TABLE_INFO[s]["impressions"], len(FRAMES[s])) for s in D.SPLITS})


All recorded requests/candidates: {'train': (232887, 2585747), 'validation': (29110, 344666), 'test': (29142, 348951)}


In [4]:
VOCAB = json.loads(PATHS["feature_vocabulary"].read_text())
CATALOGUE, CATALOGUE_META = D.load_catalogue([PATHS["articles_train_val"], PATHS["articles_test"]])
CATALOGUE_INFO = CATALOGUE_META["info"]   # ordered union: train_val then test, first record wins
SNAPSHOT_IDS = {"train_val": CATALOGUE_META["snapshot_ids"][str(PATHS["articles_train_val"])],
                "test": CATALOGUE_META["snapshot_ids"][str(PATHS["articles_test"])]}
log("Catalogue:", CATALOGUE_INFO)

def expected_dim(kind, default):
    for key in ("embedding_metadata_train_val", "embedding_metadata_test"):
        if PATHS.get(key):
            meta = json.loads(PATHS[key].read_text())
            value = meta.get(f"{kind}_dimension")
            if value and value != default:
                raise ValueError(f"{key}: {kind} dimension {value} != {default}")
    return default
TEXT_DIM, IMAGE_DIM = expected_dim("text", 768), expected_dim("image", 1024)

VECTOR_INFO, VECTOR_SOURCES = {}, {"text": [], "image": []}
for kind, column, flag, dim in (("text", "text_embedding", "has_text_embedding", TEXT_DIM),
                                ("image", "image_embedding", "has_image_embedding", IMAGE_DIM)):
    for scope in ("train_val", "test"):
        ids, matrix, available, info = D.decode_vectors(PATHS[f"{kind}_{scope}"], column, flag, dim)
        VECTOR_SOURCES[kind].append((ids, matrix, available))
        VECTOR_INFO[f"{kind}_{scope}"] = info
        log(f"{kind:5s} {scope:9s}", {k: v for k, v in info.items() if k != "path"})
TEXT_LOOKUP, text_union = D.union_vectors(VECTOR_SOURCES["text"])
IMAGE_LOOKUP, image_union = D.union_vectors(VECTOR_SOURCES["image"])
for name, union in (("text", text_union), ("image", image_union)):
    VECTOR_INFO[f"{name}_union"] = {k: v for k, v in union.items() if k != "known_ids"}
    log(name, "union:", VECTOR_INFO[f"{name}_union"])
del VECTOR_SOURCES


Catalogue: {'/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/train_val/articles.parquet': {'articles': 20738, 'published_time_missing': 0}, '/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/test/articles.parquet': {'articles': 125541, 'published_time_missing': 0}, 'overlapping_ids_first_kept': 20738}
text  train_val {'rows': 20738, 'available': 20720, 'missing': 18, 'dim': 768, 'norm_min_median_max': [0.9999991059303284, 1.0, 1.0000008344650269], 'storage_type': 'list<element: float>', 'reader': 'pyarrow 23.0.1'}
text  test      {'rows': 5619, 'available': 5618, 'missing': 1, 'dim': 768, 'norm_min_median_max': [0.9999991059303284, 1.0, 1.0000008344650269], 'storage_type': 'list<element: float>', 'reader': 'pyarrow 23.0.1'}
image train_val {'rows': 20738, 'available': 16995, 'missing': 3743, 'dim': 1024, 'norm_min_median_max': [0.9999998807907104, 1.0, 1.0000001192092896], 'storage_type': 'fixed_size_list<element: float>[1024]', 'reader': 'duckdb 1.3.2

In [5]:
HISTORIES = {}
for split in D.SPLITS:
    HISTORIES[split] = D.build_history(
        PATHS[f"history_{split}"], SPLIT_START[split], HISTORY_MAX_LEN,
        users_needed=FRAMES[split]["user_id"].to_numpy())
    log(split, "history:", HISTORIES[split].stats)


train history: {'history_users': 15143, 'events_total': 2426247, 'events_dropped_null_id_or_time': 0, 'read_time_missing': 0, 'scroll_missing': 255076, 'max_event_time': '2023-05-18T06:59:59', 'split_start': '2023-05-18T07:00:01', 'max_len': 50, 'users_truncated': 10067, 'users_empty': 0, 'length_quantiles_before_truncation': {0.0: 5.0, 0.5: 93.0, 0.9: 414.0, 1.0: 1896.0}, 'behaviour_users_without_history_row': 0}
validation history: {'history_users': 1719, 'events_total': 252567, 'events_dropped_null_id_or_time': 0, 'read_time_missing': 0, 'scroll_missing': 24847, 'max_event_time': '2023-05-25T06:59:56', 'split_start': '2023-05-25T07:00:15', 'max_len': 50, 'users_truncated': 1121, 'users_empty': 0, 'length_quantiles_before_truncation': {0.0: 5.0, 0.5: 83.0, 0.9: 372.2000000000003, 1.0: 960.0}, 'behaviour_users_without_history_row': 0}
test history: {'history_users': 1757, 'events_total': 262909, 'events_dropped_null_id_or_time': 0, 'read_time_missing': 0, 'scroll_missing': 25269, 'max

## 4. Prepare inputs and the selection sample

The article encoder, 50-event chronological histories, missing-vector handling,
metadata vocabulary and train-fitted transformations match the initial model.
Every split uses the same ordered catalogue union and publication-time policy.
Missing vectors do not remove an otherwise eligible article.

The fixed seed-42 selection sample contains 2,000 validation requests, selected
by the same request-hash rule used in the ablation's exclusion sample. Neither
selection nor exported recommendations use invented catalogue non-click labels.

In [6]:
def referenced_ids(split):
    hist = HISTORIES[split]
    mask = np.arange(hist.article_ids.shape[1])[None, :] < hist.lengths[:, None]
    return np.concatenate([FRAMES[split]["candidate_article_id"].to_numpy(), hist.article_ids[mask]])

all_ids = np.concatenate([referenced_ids(s) for s in D.SPLITS] + [CATALOGUE.index.to_numpy()])
train_ids = referenced_ids("train")
article_stats = D.fit_article_stats(CATALOGUE, train_ids)
ID_VOCAB = {}  # learned article IDs are off; metadata/vectors support unseen articles
ARTICLES, ARTICLE_INFO = D.build_article_table(
    CATALOGUE, all_ids, VOCAB, article_stats, TEXT_LOOKUP, IMAGE_LOOKUP,
    TEXT_DIM, IMAGE_DIM, 8, ID_VOCAB)
PREPROCESSING = {
    "numeric_stats": D.fit_numeric_stats(FRAMES["train"], HISTORIES["train"], OFFSETS["train"], ARTICLES),
    "article_stats": article_stats, "history_max_len": HISTORY_MAX_LEN,
    "device_columns": DEVICE_COLUMNS, "feature_vocabulary": VOCAB,
    "text_dim": TEXT_DIM, "image_dim": IMAGE_DIM, "max_topics": 8,
    "candidate_features": D.CANDIDATE_FEATURE_NAMES,
}
ARRAYS = {s: D.build_split_arrays(s, FRAMES[s], OFFSETS[s], HISTORIES[s], ARTICLES,
                                PREPROCESSING["numeric_stats"], DEVICE_COLUMNS) for s in D.SPLITS}
coverage = {s: D.coverage_report(ARRAYS[s], ARTICLES) for s in D.SPLITS}
for a in ARRAYS.values():
    assert np.isfinite(a.imp_context).all() and np.isfinite(a.cand_features).all()
    assert np.isfinite(a.hist_features).all()
assert ARRAYS["train"].event_names[M.GAP_PREV_FEATURE] == "log_gap_prev_h_z"
assert ARRAYS["train"].event_names[M.GAP_PREV_MISSING] == "gap_prev_missing"
log("Article inputs:", ARTICLE_INFO)
log("Feature order:", {"context": ARRAYS["train"].context_names,
                        "candidate": ARRAYS["train"].candidate_names,
                        "event": ARRAYS["train"].event_names,
                        "article_numeric": ARTICLES.numeric_names})


Article inputs: {'articles': 125541, 'not_in_catalogue': 0, 'category_unknown': 52, 'category_missing': 0, 'type_unknown': 12, 'published_time_missing': 0, 'with_text_vector': 20997, 'with_image_vector': 17243}
Feature order: {'context': ['hour_sin', 'hour_cos', 'dow_0', 'dow_1', 'dow_2', 'dow_3', 'dow_4', 'dow_5', 'dow_6', 'is_sso_user', 'is_sso_user_missing', 'is_subscriber', 'is_subscriber_missing', 'is_front_page', 'is_front_page_missing', 'ohe_device_type_000', 'ohe_device_type_001', 'ohe_device_type_002', 'ohe_device_type_missing', 'ohe_device_type_unknown', 'log_history_length_z', 'history_empty'], 'candidate': ['log_age_h_z', 'age_missing'], 'event': ['log_read_time_z', 'read_time_missing', 'scroll_z', 'scroll_missing', 'log_gap_prev_h_z', 'gap_prev_missing', 'log_article_age_at_read_z', 'age_at_read_missing'], 'article_numeric': ['log_body_words_z', 'body_words_missing', 'sentiment_z', 'sentiment_missing', 'premium', 'premium_missing', 'in_catalogue', 'has_text_vector', 'has_i

In [7]:
UNION_IDS = CATALOGUE.index.to_numpy(dtype=np.int64)
POLICY = D.EligibilityPolicy(["train_val/articles.parquet", "test/articles.parquet"])
INDEX = D.build_catalogue_index("all", "union", UNION_IDS, ARTICLES)
ELIGIBILITY = {s: D.eligibility_summary(ARRAYS[s], INDEX) for s in D.SPLITS}
CLICK_ELIGIBILITY = {s: D.clicked_eligibility(ARRAYS[s], INDEX, ARTICLES) for s in D.SPLITS}
VAL = ARRAYS["validation"]
SELECTION_SAMPLE, selection_bids = D.select_requests(
    VAL.user_id, VAL.impression_id, VAL.imp_time_us, "validation", SEED, SELECTION_REQUESTS)
SELECTION = {
    "objective": SELECTION_OBJECTIVE, "tiebreak": SELECTION_TIEBREAK,
    "earlier_checkpoint_on_equal_metrics": True, "interval": "every epoch",
    "selection_k": SELECTION_K, "pool_policy": POLICY.to_json(),
    "freshness_grid": list(FRESHNESS_GRID), "within_epoch_tie": "smaller freshness weight",
    "reserved_final_partition": "labelled project test; outcomes deferred until full pipeline freeze",
    "sampling": "smallest BLAKE2b-64(seed:split:behaviour_id), then behaviour_id; chronological output",
    "seed": SEED, "requests": selection_bids, "population_requests": VAL.n_impressions,
    "logged_metrics": [SELECTION_OBJECTIVE, SELECTION_TIEBREAK,
                       "eligible_only_recall@100", "diagnostic_recorded_slate_auc", "freshness_weight"],
}
(RUN_DIR / "selection").mkdir()
D.write_json(SELECTION, RUN_DIR / "selection" / "declaration.json")
pd.DataFrame({"behaviour_id": selection_bids}).to_parquet(
    RUN_DIR / "selection" / "validation_requests.parquet", index=False)
CATALOGUE_FINGERPRINT = hashlib.sha256(UNION_IDS.astype("<i8").tobytes()).hexdigest()
log("Complete pools and scoring work:", ELIGIBILITY)
# Conservative uncompressed estimate for retained score/list rows plus request manifests.
OUTPUT_ESTIMATE_BYTES = 0
for s, a in ARRAYS.items():
    lo, hi = INDEX.bounds(a.imp_time_us)
    retained = int(np.minimum(EXPORT_K, hi - lo).sum())
    estimate = retained * 64 + a.n_candidates * 48 + a.n_impressions * 64
    OUTPUT_ESTIMATE_BYTES += estimate
    log(s, "eligible pairs", ELIGIBILITY[s]["eligible_pairs"], "retained rows", retained,
        "approx uncompressed output GB", round(estimate / 1e9, 3))
log("Output estimate GB:", round(OUTPUT_ESTIMATE_BYTES / 1e9, 3),
    "| local free GB:", round(shutil.disk_usage(RUN_DIR).free / 1e9, 1),
    "| mounted Drive free GB:", round(shutil.disk_usage(PERSIST_RUN_DIR).free / 1e9, 1),
    "(mounted free space may differ from the account quota)")
log("Scoring is exhaustive and may take hours; progress reports runtime/ETA during export.",
    "Catalogue articles without vectors are retained.")
copy_tree(RUN_DIR / "selection", PERSIST_RUN_DIR / "selection")

Complete pools and scoring work: {'train': {'split': 'train', 'snapshot': 'union', 'snapshot_articles': 125541, 'snapshot_excluded_missing_published_time': 0, 'snapshot_with_text_vector': 20997, 'snapshot_with_image_vector': 17243, 'requests': 232887, 'eligible_pairs': 28552935815, 'requests_in_split': 232887, 'empty_pools': 0, 'pool_size_min_median_max': [121981, 122557.0, 123393], 'pair_share_with_text_vector': 0.15265572486981055, 'pair_share_with_image_vector': 0.1275198129394177, 'max_age_hours': None}, 'validation': {'split': 'validation', 'snapshot': 'union', 'snapshot_articles': 125541, 'snapshot_excluded_missing_published_time': 0, 'snapshot_with_text_vector': 20997, 'snapshot_with_image_vector': 17243, 'requests': 29110, 'eligible_pairs': 3608174696, 'requests_in_split': 29110, 'empty_pools': 0, 'pool_size_min_median_max': [123393, 123943.0, 124472], 'pair_share_with_text_vector': 0.15829812276916427, 'pair_share_with_image_vector': 0.13111968539784916, 'max_age_hours': None}

In [8]:
ARTICLE_TENSORS = M.article_table_tensors(ARTICLES, DEVICE)
TENSORS = {s: M.SplitTensors(ARRAYS[s], DEVICE) for s in D.SPLITS}
VAL_CLICK_LOOKUP = D.clicks_by_request(CLICK_ELIGIBILITY["validation"])
selection_slates = D.subset_split_arrays(VAL, SELECTION_SAMPLE)
SELECTION_TENSORS = M.SplitTensors(selection_slates, DEVICE)

def model_config(overrides=None):
    sample = ARRAYS["train"]
    gap = PREPROCESSING["numeric_stats"]["log_gap_to_impression_h"]
    params = {**MODEL_HPARAMS, **(overrides or {})}
    return M.ModelConfig(
        variant="attention", n_categories=len(VOCAB["category"]) + 3,
        n_types=len(VOCAB["article_type"]) + 3, n_topics=len(VOCAB["topics"]) + 2,
        n_article_numeric=ARTICLES.numeric.shape[1], text_dim_in=TEXT_DIM, image_dim_in=IMAGE_DIM,
        n_id_buckets=1, n_event_features=sample.hist_features.shape[-1],
        n_candidate_features=sample.cand_features.shape[1], n_context_features=sample.imp_context.shape[1],
        gap_mean=gap["mean"], gap_std=gap["std"], **params)

## 5. Benchmark inference and train one model

First compare full eligible logits and top-100/200 rankings for 32 declared
selection requests using the previous saved weights. The benchmark records
numerical agreement and scoring/top-k timings without claiming a hardware-independent
speedup. Caches are built before timing. All inference remains float32.

Then train a fresh seed-42 model using the initial study's selected neural
hyperparameters. At each epoch, score the selection catalogue once and evaluate
all nine freshness weights from those same logits. Optuna records the grid
trials; there is no separate neural fit per weight. Checkpoint selection and the
learning-rate/early-stopping decisions use the best blended Recall@100, with
nDCG@100 as the tiebreak. The final inference bundle saves the chosen weight
alongside the selected neural checkpoint.

In [9]:
# Benchmark only on Colab; preparation of this notebook did not run these models.
benchmark_requests = SELECTION_SAMPLE[np.linspace(0, len(SELECTION_SAMPLE) - 1, 32, dtype=int)]
reference_model = M.build_model(model_config(), ARTICLE_TENSORS, DEVICE)
reference_model.load_state_dict(torch.load(REFERENCE_DIR / "model" / "model_state.pt",
                                          map_location=DEVICE, weights_only=True))
reference_model.eval()
reference_scorer = M.CatalogueScorer(reference_model, TENSORS["validation"], INDEX,
    PREPROCESSING["numeric_stats"]["log_candidate_age_h"], requests=benchmark_requests)
batched_scorer = F.BatchedCatalogueScorer(reference_model, TENSORS["validation"], INDEX,
    PREPROCESSING["numeric_stats"]["log_candidate_age_h"], PAIR_CHUNK, REQUEST_BATCH,
    requests=benchmark_requests)
BENCHMARK = F.benchmark_equivalence(reference_scorer, batched_scorer,
                                    benchmark_requests, RELOAD_TOLERANCE)
D.write_json(BENCHMARK, RUN_DIR / "inference_benchmark.json")
D._copy_atomic(RUN_DIR / "inference_benchmark.json", PERSIST_RUN_DIR / "inference_benchmark.json")
log("Saved-model inference benchmark:", BENCHMARK)
del reference_model, reference_scorer, batched_scorer

TUNING_DIR = RUN_DIR / "tuning"
TUNING_DIR.mkdir()
live_db = LOCAL_ROOT / f"freshness_{RUN_ID}.db"
study = optuna.create_study(study_name=f"freshness_{RUN_ID}", direction="maximize",
                            storage=f"sqlite:///{live_db}")
SELECTION_CURVES = []

def save_tuning():
    study.trials_dataframe().to_csv(TUNING_DIR / "trials.csv", index=False)
    with sqlite3.connect(live_db) as source_db, sqlite3.connect(TUNING_DIR / "study.db") as target_db:
        source_db.backup(target_db)
    D.write_json(SELECTION_CURVES, TUNING_DIR / "selection_curves.json")
    copy_tree(TUNING_DIR, PERSIST_RUN_DIR / "tuning")

def select_validation(m):
    epoch = len(SELECTION_CURVES) + 1
    scorer = F.BatchedCatalogueScorer(m, TENSORS["validation"], INDEX,
        PREPROCESSING["numeric_stats"]["log_candidate_age_h"], PAIR_CHUNK, REQUEST_BATCH,
        requests=SELECTION_SAMPLE)
    hits = {w: {} for w in FRESHNESS_GRID}
    t0 = time.time()
    for block, counts, outputs in scorer.batches(SELECTION_SAMPLE, SELECTION_K, FRESHNESS_GRID):
        for weight, output in outputs.items():
            for row, i in enumerate(block):
                ids = output["article_ids"][row, :min(SELECTION_K, int(counts[row]))]
                hit = D.click_hits(VAL_CLICK_LOOKUP, int(i), ids, (SELECTION_K,))
                if hit is not None:
                    hits[weight][int(i)] = hit
    curve = []
    for weight in FRESHNESS_GRID:
        metrics = D.recall_from_hits(hits[weight], CLICK_ELIGIBILITY["validation"],
                                     (SELECTION_K,), requests_scored=len(SELECTION_SAMPLE))
        study.enqueue_trial({"freshness_weight": weight})
        trial = study.ask()
        trial.suggest_categorical("freshness_weight", list(FRESHNESS_GRID))
        trial.set_user_attr("epoch", epoch)
        trial.set_user_attr(SELECTION_TIEBREAK, metrics[SELECTION_TIEBREAK])
        trial.set_user_attr("neural_seed", SEED)
        study.tell(trial, metrics[SELECTION_OBJECTIVE])
        curve.append({**metrics, "freshness_weight": weight, "optuna_trial": trial.number})
    chosen = max(curve, key=lambda row: (row[SELECTION_OBJECTIVE], row[SELECTION_TIEBREAK],
                                          -row["freshness_weight"]))
    SELECTION_CURVES.append({"epoch": epoch, "seconds": time.time() - t0,
                             "chosen_weight": chosen["freshness_weight"], "curve": curve})
    save_tuning()
    log("Freshness selection epoch", epoch, "weight", chosen["freshness_weight"],
        "Recall@100", chosen[SELECTION_OBJECTIVE])
    diagnostic_scores = F.recorded_scores(M.predict_logits(m, SELECTION_TENSORS),
        selection_slates, ARTICLES, chosen["freshness_weight"])
    diagnostic = D.impression_metrics(selection_slates.offsets, selection_slates.labels,
                                       diagnostic_scores, k=5)
    return {**chosen, "diagnostic_recorded_slate_auc": diagnostic["mean_impression_auc"]}

Saved-model inference benchmark: {'requests': 32, 'max_abs_logit_difference': 0.0, 'tolerance': 0.0001, 'top100_top200_match_at_weights_0_and_2': True, 'reference_score_seconds': 0.21986958700000514, 'batched_score_seconds': 0.13772288699999535, 'score_only_speedup': 1.5964636799983183, 'reference_top200_seconds': 0.2738687120000236, 'batched_top200_seconds': 0.14567585900005042, 'top200_speedup': 1.879986937299809, 'passed': True}


[I 2026-11-04 10:00:48,937] A new study created in RDB with name: freshness_20261104_175920_improved_seed42


In [10]:
MAIN_CONFIG = model_config()
TRAIN_CONFIG = M.TrainConfig(**{**TRAIN_HPARAMS, "seed": SEED})
assert TRAIN_CONFIG.max_train_impressions_per_epoch is None
M.set_seed(SEED)
model = M.build_model(MAIN_CONFIG, ARTICLE_TENSORS, DEVICE)
TRAINING_DIR = RUN_DIR / "model_training"

def persist_epoch(epoch, row):
    copy_tree(TRAINING_DIR, PERSIST_RUN_DIR / "model_training")

t0 = time.time()
HISTORY, BEST = M.train_model(model, TENSORS["train"], select_validation, SELECTION,
    TRAIN_CONFIG, TRAINING_DIR, resume=False, on_epoch_end=persist_epoch, log=log)
TRAIN_SECONDS = time.time() - t0
model.eval()  # train_model restored best_model_state.pt, not the last epoch
D.write_json(HISTORY, RUN_DIR / "training_history.json")
pd.DataFrame(HISTORY).to_csv(RUN_DIR / "training_history.csv", index=False)
FRESHNESS_WEIGHT = float(BEST["metrics"]["freshness_weight"])
TUNING_SUMMARY = {
    "method": "Optuna-recorded fixed freshness grid at each epoch; one neural fit",
    "reference_run_id": REFERENCE_RUN_ID, "reference_neural_tuning": REFERENCE_TUNING,
    "freshness_grid": list(FRESHNESS_GRID), "selection_requests": len(SELECTION_SAMPLE),
    "actual_trials": len(study.trials), "completed_trials": len(study.trials),
    "best_epoch": BEST["epoch"], "best_freshness_weight": FRESHNESS_WEIGHT,
    "objective": SELECTION_OBJECTIVE, "tiebreak": SELECTION_TIEBREAK,
}
D.write_json(TUNING_SUMMARY, TUNING_DIR / "summary.json")
save_tuning()
log("Restored selected epoch", BEST["epoch"], "validation selection key", BEST["key"],
    "| final-fit requests per epoch:", ARRAYS["train"].n_impressions)

/content/lstm_improved_code/lstm_model.py:598: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total += float(loss) * len(batch["labels"])


  epoch 1 step 200/910 loss 0.2986 (7s)
  epoch 1 step 400/910 loss 0.2918 (13s)
  epoch 1 step 600/910 loss 0.2884 (18s)
  epoch 1 step 800/910 loss 0.2862 (23s)
Freshness selection epoch 1 weight 4.0 Recall@100 0.8849
epoch 1: train BCE 0.2852 | val known_click_recall@100 0.8849 observed_click_ndcg@100 0.2488 | diag AUC 0.5629 | lr 1.53e-03 | 42s *best*
  epoch 2 step 200/910 loss 0.2758 (6s)
  epoch 2 step 400/910 loss 0.2756 (11s)
  epoch 2 step 600/910 loss 0.2751 (16s)
  epoch 2 step 800/910 loss 0.2747 (22s)
Freshness selection epoch 2 weight 4.0 Recall@100 0.8859
epoch 2: train BCE 0.2744 | val known_click_recall@100 0.8859 observed_click_ndcg@100 0.2524 | diag AUC 0.5685 | lr 1.53e-03 | 41s *best*
  epoch 3 step 200/910 loss 0.2720 (5s)
  epoch 3 step 400/910 loss 0.2717 (11s)
  epoch 3 step 600/910 loss 0.2710 (16s)
  epoch 3 step 800/910 loss 0.2707 (22s)
Freshness selection epoch 3 weight 4.0 Recall@100 0.8854
epoch 3: train BCE 0.2708 | val known_click_recall@100 0.8854 ob

## 6. Selection results

The executed history reports the selected freshness weight and blended retrieval
metrics at each epoch. All 2,000 requests are reused for tuning and checkpoint
selection, so these are development results. The best checkpoint is restored
before any full-validation or project-test exports.

In [11]:
log("Optuna outcome:", TUNING_SUMMARY)
log("Selected checkpoint sample metrics:", BEST["metrics"])
log("Selected model:", MAIN_CONFIG.to_json())
display(pd.DataFrame(HISTORY))


Optuna outcome: {'method': 'Optuna-recorded fixed freshness grid at each epoch; one neural fit', 'reference_run_id': '20261104_014914_seed42', 'reference_neural_tuning': {'study_name': 'lstm_recall100_20261104_014914_seed42', 'objective': 'known_click_recall@100', 'tiebreak': 'observed_click_ndcg@100', 'final_tie_rule': 'earlier trial', 'seed_each_trial': 42, 'search_space': {'lstm_hidden': [64, 128, 256], 'attention_dim': [32, 64, 128], 'article_dim': [64, 128], 'dropout': [0.1, 0.4], 'learning_rate_log': [0.0003, 0.003], 'weight_decay_log': [1e-06, 0.001], 'history_dropout': [0.0, 0.3]}, 'sampler': 'TPESampler(seed=42)', 'pruner': 'MedianPruner(startup=3, warmup_steps=1)', 'budget': {'max_trials': 12, 'timeout_min': 150, 'max_epochs': 3, 'train_requests_per_epoch': 60000}, 'actual_trials': 12, 'completed_trials': 5, 'trial_states': {'COMPLETE': 5, 'PRUNED': 7}, 'best_trial': 0, 'best_params': {'lstm_hidden': 128, 'attention_dim': 32, 'article_dim': 128, 'dropout': 0.2803345035229627,

,epoch,train_bce,known_click_recall@100,observed_click_ndcg@100,eligible_only_recall@100,diagnostic_recorded_slate_auc,freshness_weight,learning_rate,train_seconds,selection_seconds,seconds
0,1,0.285186,0.8849,0.248839,0.8849,0.562856,4.0,0.001532,26.313974,16.164430,42.478602
1,2,0.274386,0.8859,0.252428,0.8859,0.568548,4.0,0.001532,24.846120,16.190125,41.036466
2,3,0.270845,0.8854,0.255227,0.8854,0.571883,4.0,0.001532,24.524783,16.115618,40.640609
3,4,0.267248,0.8899,0.255759,0.8899,0.574460,4.0,0.000766,24.826054,16.127771,40.954026
4,5,0.265672,0.8864,0.246320,0.8864,0.546031,8.0,0.000766,24.716608,16.130421,40.847234
5,6,0.263642,0.8909,0.274564,0.8909,0.622791,2.0,0.000383,24.830712,16.148508,40.979419
6,7,0.262816,0.8879,0.258980,0.8879,0.583347,4.0,0.000383,24.999089,16.217657,41.216941
7,8,0.261544,0.8874,0.276177,0.8874,0.626587,2.0,0.000191,25.119093,16.139470,41.258786


## 7. Save the score recipe and export every split

Persist the selected weights, fitted transformations, named features, article
inputs and freshness recipe before the long export. Then score every eligible
catalogue article for every train, validation and project-test request in bounded
request/candidate blocks. Retain the exact top 200, using descending blended
score and ascending article ID. No slate or clicked items are inserted.

Only complete local Parquet files are copied to Drive. Catalogue scores and
recorded-slate scores use the same final score meaning. They are ranking scores,
not calibrated click probabilities. Full validation is reported at 100; the
project-test labels remain masked and no test outcome metrics are calculated.

In [12]:
MODEL_DIR = RUN_DIR / "model"
(MODEL_DIR / "code").mkdir(parents=True)
M.atomic_torch_save(model.state_dict(), MODEL_DIR / "model_state.pt")
D.write_json(MAIN_CONFIG.to_json(), MODEL_DIR / "model_config.json")
D.write_json(PREPROCESSING, MODEL_DIR / "preprocessing.json")
feature_dictionary = {
    "model_family": "Single LSTM with target-aware attention and tuned freshness",
    "ordered_features": {"request_context": ARRAYS["train"].context_names,
                         "candidate": ARRAYS["train"].candidate_names,
                         "history_event": ARRAYS["train"].event_names,
                         "article_numeric": ARTICLES.numeric_names},
    "category_index_to_value": {**{"0": "padding", "1": "missing", "2": "unknown"},
        **{str(i + 3): v for i, v in enumerate(VOCAB["category"])}},
    "article_type_index_to_value": {**{"0": "padding", "1": "missing", "2": "unknown"},
        **{str(i + 3): v for i, v in enumerate(VOCAB["article_type"])}},
    "topic_index_to_value": {**{"0": "padding", "1": "unknown"},
        **{str(i + 2): v for i, v in enumerate(VOCAB["topics"])}},
    "fitted_transformations": "preprocessing.json; reuse fitted state without refitting",
    "article_row_to_original_id": "mappings.npz: article_table_ids",
    "frozen_article_inputs": "article_inputs.npz; compact frozen vectors, metadata, missingness flags",
    "history_order": "oldest first in retained fixed history; masked padding",
    "missing_vectors": "zero vector placeholder plus has_text_vector/has_image_vector flags",
    "score": "neural_logit - freshness_weight * log1p(age_hours); ranking score, not a probability",
}
D.write_json(feature_dictionary, MODEL_DIR / "feature_dictionary.json")
D.write_json({"policy": POLICY.to_json(), "selection_k": SELECTION_K, "export_k": EXPORT_K,
              "shortlist_size": SHORTLIST_SIZE, "final_display_size": FINAL_DISPLAY_SIZE},
             MODEL_DIR / "catalogue_policy.json")
np.savez_compressed(MODEL_DIR / "mappings.npz", article_table_ids=ARTICLES.article_ids,
                    train_referenced_ids=np.unique(train_ids), catalogue_source_ids=UNION_IDS,
                    id_vocab_ids=np.array([], dtype=np.int64), id_vocab_rows=np.array([], dtype=np.int64))
# Include the frozen lookup so inference does not depend on rebuilding vector snapshots.
np.savez_compressed(MODEL_DIR / "article_inputs.npz",
                    **{f.name: np.asarray(getattr(ARTICLES, f.name)) for f in fields(D.ArticleTable)})
for name in ("lstm_model.py", "lstm_data.py", "lstm_inference.py"):
    shutil.copy2(CODE_DIR / name, MODEL_DIR / "code" / name)
(RUN_DIR / "requirements.txt").write_text("\n".join(f"{k}=={v}" for k, v in VERSIONS.items()) + "\n")
SCORE_RECIPE = {"formula": "neural_logit - freshness_weight * log1p(age_hours)",
    "freshness_weight": FRESHNESS_WEIGHT, "selected_epoch": BEST["epoch"],
    "age_units": "hours from integer-second request and publication timestamps, clamped at zero",
    "eligibility": "shared publication-time policy, compared in microseconds",
    "score_semantics": "uncalibrated blended ranking score; higher is better",
    "selection_requests": len(SELECTION_SAMPLE), "neural_reference_run": REFERENCE_RUN_ID}
D.write_json(SCORE_RECIPE, MODEL_DIR / "score_recipe.json")
copy_tree(MODEL_DIR, PERSIST_RUN_DIR / "model")
for name in ("training_history.json", "training_history.csv", "requirements.txt", "reference_hyperparameters.json", "split_manifest.json"):
    D._copy_atomic(RUN_DIR / name, PERSIST_RUN_DIR / name)
log("Selected inference bundle saved to", PERSIST_RUN_DIR / "model")

Selected inference bundle saved to /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/2.1.3_improved_run/20261104_175920_improved_seed42/model


In [2]:
# Every original recorded candidate, including shown non-clicks and ineligible items.
DIAGNOSTICS, VALIDATION_SLATE_METRICS = {}, None
DIAG_SCHEMA = pa.schema([("behaviour_id", pa.string()), ("article_id", pa.int64()),
                         ("score", pa.float32())])
for split in D.SPLITS:
    a = ARRAYS[split]
    scores = M.predict_logits(model, TENSORS[split], batch_size=TRAIN_CONFIG.eval_batch_impressions)
    if split == "validation":
        RAW_NEURAL_SLATE_METRICS = D.impression_metrics(a.offsets, a.labels, scores, k=5)
    scores = F.recorded_scores(scores, a, ARTICLES, FRESHNESS_WEIGHT)
    assert np.isfinite(scores).all()
    bids = np.repeat(D.behaviour_ids(a.user_id, a.impression_id), np.diff(a.offsets))
    path = RUN_DIR / "diagnostics" / "recorded_slates" / f"{split}_scores.parquet"
    path.parent.mkdir(parents=True, exist_ok=True)
    with pq.ParquetWriter(path, DIAG_SCHEMA, compression="zstd") as writer:
        for start in range(0, len(scores), 100_000):
            stop = min(start + 100_000, len(scores))
            writer.write_table(pa.Table.from_pydict({"behaviour_id": bids[start:stop],
                "article_id": a.cand_article_id[start:stop], "score": scores[start:stop].astype(np.float32)},
                schema=DIAG_SCHEMA))
    DIAGNOSTICS[split] = {"path": str(path.relative_to(RUN_DIR)), "rows": len(scores),
                          "requests": a.n_impressions, "labels_joined_separately": True}
    if split == "validation":
        VALIDATION_SLATE_METRICS = D.impression_metrics(a.offsets, a.labels, scores, k=5)
    D._copy_atomic(path, PERSIST_RUN_DIR / path.relative_to(RUN_DIR))
    log(split, "recorded-slate diagnostic rows", len(scores))
    del bids, scores

train recorded-slate diagnostic rows 2585747
validation recorded-slate diagnostic rows 344666
test recorded-slate diagnostic rows 348951


In [3]:
CATALOGUE_RECORDS = {}
for split in D.SPLITS:
    a = ARRAYS[split]
    bids = D.behaviour_ids(a.user_id, a.impression_id)
    scorer = F.BatchedCatalogueScorer(model, TENSORS[split], INDEX,
        PREPROCESSING["numeric_stats"]["log_candidate_age_h"], PAIR_CHUNK, REQUEST_BATCH)
    lookup = D.clicks_by_request(CLICK_ELIGIBILITY[split])
    weights = (0., FRESHNESS_WEIGHT) if split == "validation" else (FRESHNESS_WEIGHT,)
    hits = {w: {} for w in weights}
    baseline_hits = {}
    ties = {100: 0, 200: 0}
    writer = F.ColumnarShortlistWriter(RUN_DIR, split, EXPORT_K)
    t0, next_progress = time.time(), 1000
    for block, counts, outputs in scorer.batches(np.arange(a.n_impressions), EXPORT_K, weights):
        selected = outputs[FRESHNESS_WEIGHT]
        writer.add_batch(bids[block], a.imp_time_us[block], counts,
                         selected["article_ids"], selected["scores"])
        for row, i in enumerate(block):
            count = min(EXPORT_K, int(counts[row]))
            for k in ties:
                ties[k] += int(counts[row] > k and selected["scores"].shape[1] > k
                               and selected["scores"][row, k - 1] == selected["scores"][row, k])
            for weight, output in outputs.items():
                hit = D.click_hits(lookup, int(i), output["article_ids"][row, :count], (SELECTION_K,))
                if hit is not None:
                    hits[weight][int(i)] = hit
            if split == "validation":
                lo, hi = int(scorer.lo[i]), int(scorer.hi[i])
                start = max(lo, hi - SELECTION_K)
                while start > lo and INDEX.published_us[start - 1] == INDEX.published_us[start]:
                    start -= 1
                order = np.lexsort((INDEX.article_ids[start:hi], -INDEX.published_us[start:hi]))
                recent = INDEX.article_ids[start:hi][order[:SELECTION_K]]
                hit = D.click_hits(lookup, int(i), recent, (SELECTION_K,))
                if hit is not None:
                    baseline_hits[int(i)] = hit
        done = int(block[-1]) + 1
        if done >= next_progress:
            elapsed = time.time() - t0
            log(split, done, "/", a.n_impressions, "requests | elapsed min", round(elapsed / 60, 1),
                "| ETA min", round(elapsed / done * (a.n_impressions - done) / 60, 1))
            next_progress += 1000
    files = writer.close()
    check = D.validate_catalogue_exports(a, INDEX, files, EXPORT_K)
    metrics = D.recall_from_hits(hits[FRESHNESS_WEIGHT], CLICK_ELIGIBILITY[split],
                                 (SELECTION_K,), requests_scored=a.n_impressions)
    if split == "validation":
        RAW_NEURAL_CATALOGUE_METRICS = D.recall_from_hits(hits[0.], CLICK_ELIGIBILITY[split],
            (SELECTION_K,), requests_scored=a.n_impressions)
        NEWEST_FIRST_METRICS = D.recall_from_hits(baseline_hits, CLICK_ELIGIBILITY[split],
            (SELECTION_K,), requests_scored=a.n_impressions)
    CATALOGUE_RECORDS[split] = {
        "scope": "all recorded requests", "files": {k: str(Path(files[k]).relative_to(RUN_DIR))
            for k in ("rankings", "scores", "manifest")}, "coverage": ELIGIBILITY[split],
        "metrics": metrics, "ties_crossing_cutoff": {str(k): v for k, v in ties.items()},
        "freshness_weight": FRESHNESS_WEIGHT, "seconds": time.time() - t0, "integrity_check": check,
    }
    D.write_json(CATALOGUE_RECORDS[split], RUN_DIR / "catalogue" / f"{split}.json")
    for name in ("rankings", "scores", "manifest"):
        file = Path(files[name])
        D._copy_atomic(file, PERSIST_RUN_DIR / file.relative_to(RUN_DIR))
    D._copy_atomic(RUN_DIR / "catalogue" / f"{split}.json", PERSIST_RUN_DIR / "catalogue" / f"{split}.json")
    log(split, "complete:", check, "retrieval:", metrics)
    del scorer, lookup, hits
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

train 1008 / 232887 requests | elapsed min 0.1 | ETA min 18.5
train 2000 / 232887 requests | elapsed min 0.2 | ETA min 18.5
train 3008 / 232887 requests | elapsed min 0.2 | ETA min 18.4
train 4000 / 232887 requests | elapsed min 0.3 | ETA min 18.3
train 5008 / 232887 requests | elapsed min 0.4 | ETA min 18.2
train 6000 / 232887 requests | elapsed min 0.5 | ETA min 18.2
train 7008 / 232887 requests | elapsed min 0.6 | ETA min 18.1
train 8000 / 232887 requests | elapsed min 0.6 | ETA min 18.0
train 9008 / 232887 requests | elapsed min 0.7 | ETA min 17.9
train 10000 / 232887 requests | elapsed min 0.8 | ETA min 17.8
train 11008 / 232887 requests | elapsed min 0.9 | ETA min 17.8
train 12000 / 232887 requests | elapsed min 1.0 | ETA min 17.7
train 13008 / 232887 requests | elapsed min 1.0 | ETA min 17.6
train 14000 / 232887 requests | elapsed min 1.1 | ETA min 17.5
train 15008 / 232887 requests | elapsed min 1.2 | ETA min 17.4
train 16000 / 232887 requests | elapsed min 1.3 | ETA min 17.4
t

In [15]:
METRICS = {
    "partition": "validation development; repeatedly used for tuning/checkpoint selection",
    "selection": {**SELECTION, "selected_epoch": BEST["epoch"], "selected_key": BEST["key"],
                  "selected_sample_metrics": BEST["metrics"]},
    "full_validation_catalogue": CATALOGUE_RECORDS["validation"]["metrics"],
    "full_validation_recorded_slates": VALIDATION_SLATE_METRICS,
    "full_validation_raw_neural_catalogue": RAW_NEURAL_CATALOGUE_METRICS,
    "full_validation_newest_first": NEWEST_FIRST_METRICS,
    "full_validation_raw_neural_recorded_slates": RAW_NEURAL_SLATE_METRICS,
    "definitions": {
        "known_click_recall@K": "macro hits / all original known clicks, including ineligible missed clicks",
        "observed_click_ndcg@K": "macro binary observed-click DCG / ideal over all original known clicks, capped K",
        "unknown_outcomes": "zero observed gain is retrieval bookkeeping, never a negative click label",
        "recorded_slate_auc": "diagnostic mean impression AUC on observed shown-click labels only",
        "ties": "descending score, ascending original article_id; crossing-cutoff ties reported",
    },
    "test": {"source_labelled": True, "evaluation": "deferred until pipeline freeze", "metrics": None},
}
# Uploaded code may have no Git checkout; source helpers themselves are bundled.
CODE_PROVENANCE = {"source": "uploaded helper ZIP; source modules saved in model/code", "commit": None,
                   "uncommitted_changes": None}
try:
    CODE_PROVENANCE["commit"] = subprocess.check_output(
        ["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
    CODE_PROVENANCE["uncommitted_changes"] = bool(subprocess.check_output(
        ["git", "-C", str(CODE_DIR), "status", "--porcelain"], stderr=subprocess.DEVNULL, text=True).strip())
except (subprocess.CalledProcessError, FileNotFoundError):
    pass
RUN_CONFIG = {
    "experiment_id": EXPERIMENT_ID, "experiment_stage": EXPERIMENT_STAGE, "owner": OWNER, "run_id": RUN_ID, "seed": SEED,
    "python": platform.python_version(), "packages": VERSIONS, "code": CODE_PROVENANCE,
    "input_files": INPUTS, "dataset_split": SPLIT_MANIFEST, "feature_case": FEATURE_CASE, "feature_dictionary": "model/feature_dictionary.json",
    "preprocessing": "model/preprocessing.json", "model": MAIN_CONFIG.to_json(),
    "train_config": asdict(TRAIN_CONFIG), "all_training_requests": ARRAYS["train"].n_impressions,
    "actual_final_epochs": len(HISTORY), "train_seconds": TRAIN_SECONDS,
    "chosen_checkpoint": {"path": "model/model_state.pt", "epoch": BEST["epoch"]},
    "selection": SELECTION, "tuning": TUNING_SUMMARY, "export_schema_version": D.EXPORT_SCHEMA_VERSION,
    "prediction_pool": "time_eligible_catalogue", "pool_policy": POLICY.to_json(),
    "catalogue_id_fingerprint_sha256": CATALOGUE_FINGERPRINT,
    "catalogue_sources": CATALOGUE_INFO, "vector_sources": VECTOR_INFO,
    "vector_duplicate_rule": "train_val then test, first record wins including missing vectors",
    "selection_k": SELECTION_K, "export_k": EXPORT_K, "shortlist_size": SHORTLIST_SIZE,
    "final_display_size": FINAL_DISPLAY_SIZE, "score_semantics": SCORE_RECIPE["score_semantics"],
    "score_recipe": "model/score_recipe.json", "freshness_weight": FRESHNESS_WEIGHT,
    "neural_reference_run": REFERENCE_RUN_ID, "inference_benchmark": BENCHMARK,
    "inference": {"pair_chunk": PAIR_CHUNK, "request_batch": REQUEST_BATCH,
                  "precision": "float32", "top_k": "exact score-bit order then article ID"},
    "export_request_scope": "all requests in train/validation/test", "catalogue_exports": CATALOGUE_RECORDS,
    "recorded_slate_exports": DIAGNOSTICS, "feature_coverage": coverage,
    "train_predictions": "in-sample; not out-of-fold", "test_labels": "reserved on disk for final evaluation; masked in development",
    "recovery": "epoch checkpoint under model_training/recovery; exact resumption not checked",
    "persistent_artifact_location": str(PERSIST_RUN_DIR),
}
D.write_json(METRICS, RUN_DIR / "metrics.json")
D.write_json(RUN_CONFIG, RUN_DIR / "run_config.json")
D.write_json({"export_schema_version": D.EXPORT_SCHEMA_VERSION,
              "selection_k": SELECTION_K, "export_k": EXPORT_K,
              "shortlist_size": SHORTLIST_SIZE, "final_display_size": FINAL_DISPLAY_SIZE,
              "prediction_pool": "time_eligible_catalogue", "pool_policy": POLICY.to_json(),
              "score_recipe": SCORE_RECIPE, "splits": CATALOGUE_RECORDS, "recorded_slates": DIAGNOSTICS}, RUN_DIR / "export_manifest.json")
# Large Parquets and the model bundle were copied when complete; do not copy them twice.
for file in RUN_DIR.iterdir():
    if file.is_file():
        D._copy_atomic(file, PERSIST_RUN_DIR / file.name)
# Verify saved-file coverage and sizes once; numerical verification follows.
FILE_COVERAGE = {str(f.relative_to(RUN_DIR)): f.stat().st_size
                 for f in RUN_DIR.rglob("*") if f.is_file()}
assert all((PERSIST_RUN_DIR / name).is_file() and (PERSIST_RUN_DIR / name).stat().st_size == size
           for name, size in FILE_COVERAGE.items())
D.write_json(FILE_COVERAGE, RUN_DIR / "manifest.json")
D._copy_atomic(RUN_DIR / "manifest.json", PERSIST_RUN_DIR / "manifest.json")
log("Full validation development metrics:", METRICS["full_validation_catalogue"])
log("Persistent file coverage/sizes checked:", len(FILE_COVERAGE), "files")

Full validation development metrics: {'requests_with_known_clicks_evaluated': 29110, 'requests_without_known_clicks_scored': 0, 'known_clicks_evaluated': 29249, 'known_clicks_eligible': 29249, 'split_click_eligibility': {'clicked_articles': 29249, 'eligible': 29249, 'eligibility_coverage': 1.0, 'not_in_catalogue_snapshot': 0, 'published_time_missing': 0, 'published_after_request': 0}, 'known_click_recall@100': 0.8969689682812321, 'observed_click_ndcg@100': 0.2752488561398548, 'eligible_only_recall@100': 0.8969689682812321, 'eligible_only_requests@100': 29110}
Persistent file coverage/sizes checked: 44 files


## 8. Reload the complete final score

Load all three helper modules, the selected weights, article inputs, feature
mappings, fitted transformations and freshness recipe from shared Drive.
Reconstruct a small validation batch without refitting. Compare both raw neural
logits and final blended scores, then verify retained catalogue IDs/scores
against the persisted validation Parquets. This checks the ranking recipe as
well as the network weights.

In [16]:
bundle = PERSIST_RUN_DIR / "model"
def load_bundled_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

BD = load_bundled_module("lstm_data", bundle / "code" / "lstm_data.py")
BM = load_bundled_module("lstm_model", bundle / "code" / "lstm_model.py")
BF = load_bundled_module("lstm_inference", bundle / "code" / "lstm_inference.py")
saved_recipe = json.loads((bundle / "score_recipe.json").read_text())
assert saved_recipe == SCORE_RECIPE
prep = json.loads((bundle / "preprocessing.json").read_text())
cfg = BM.ModelConfig.from_json(json.loads((bundle / "model_config.json").read_text()))
with np.load(bundle / "article_inputs.npz", allow_pickle=False) as saved:
    article_fields = {f.name: saved[f.name] for f in fields(BD.ArticleTable)}
article_fields["numeric_names"] = article_fields["numeric_names"].tolist()
reload_articles = BD.ArticleTable(**article_fields)
with np.load(bundle / "mappings.npz", allow_pickle=False) as mappings:
    assert np.array_equal(mappings["article_table_ids"], reload_articles.article_ids)
assert json.loads((bundle / "feature_dictionary.json").read_text()) == feature_dictionary
reloaded = BM.build_model(cfg, BM.article_table_tensors(reload_articles, DEVICE), DEVICE)
reloaded.load_state_dict(torch.load(bundle / "model_state.pt", map_location=DEVICE, weights_only=True))
reloaded.eval()
# Reconstruct only a small complete-slate batch using the persisted fitted transforms.
picks = np.arange(min(8, VAL.n_impressions))
probe_frame, probe_offsets = BD.subset_requests_frame(FRAMES["validation"], OFFSETS["validation"], picks)
probe_arrays = BD.build_split_arrays("validation", probe_frame, probe_offsets, HISTORIES["validation"],
    reload_articles, prep["numeric_stats"], prep["device_columns"])
original_arrays = D.subset_split_arrays(ARRAYS["validation"], picks)
expected_logits = M.predict_logits(model, M.SplitTensors(original_arrays, DEVICE))
loaded_logits = BM.predict_logits(reloaded, BM.SplitTensors(probe_arrays, DEVICE))
difference = np.abs(expected_logits - loaded_logits)
expected_scores = F.recorded_scores(expected_logits, original_arrays, ARTICLES, FRESHNESS_WEIGHT)
loaded_scores = BF.recorded_scores(loaded_logits, probe_arrays, reload_articles, saved_recipe["freshness_weight"])
score_difference = float(np.abs(expected_scores - loaded_scores).max())
with np.load(bundle / "mappings.npz", allow_pickle=False) as mappings:
    reload_index = BD.build_catalogue_index("all", "union", mappings["catalogue_source_ids"], reload_articles)
reload_scorer = BF.BatchedCatalogueScorer(reloaded, BM.SplitTensors(probe_arrays, DEVICE), reload_index,
    prep["numeric_stats"]["log_candidate_age_h"], PAIR_CHUNK, REQUEST_BATCH)
rank_path = PERSIST_RUN_DIR / "rankings" / "validation" / "submission_top_200.parquet"
expected_ids = next(pq.ParquetFile(rank_path).iter_batches(batch_size=len(picks))).to_pydict()["article_ids"]
score_path = PERSIST_RUN_DIR / "predictions" / "validation" / "submission_article_scores.parquet"
expected_pairs = next(pq.ParquetFile(score_path).iter_batches(batch_size=len(picks) * EXPORT_K)).to_pandas()
catalogue_difference, ranking_matches = 0., True
for block, counts, output in reload_scorer.batches(np.arange(len(picks)), EXPORT_K, (FRESHNESS_WEIGHT,)):
    for row, i in enumerate(block):
        count = min(EXPORT_K, int(counts[row]))
        ids = output[FRESHNESS_WEIGHT]["article_ids"][row, :count]
        values = output[FRESHNESS_WEIGHT]["scores"][row, :count]
        ranking_matches = ranking_matches and ids.tolist() == expected_ids[int(i)]
        behaviour_id = BD.behaviour_ids(probe_arrays.user_id[i:i+1], probe_arrays.impression_id[i:i+1])[0]
        saved_values = expected_pairs.loc[expected_pairs.behaviour_id == behaviour_id, "score"].to_numpy()
        if count:
            catalogue_difference = max(catalogue_difference, float(np.abs(values - saved_values).max()))
RELOAD_CHECK = {"source": str(bundle), "requests": len(picks), "candidate_rows": len(difference),
    "max_abs_logit_difference": float(difference.max()), "tolerance": RELOAD_TOLERANCE,
    "fitted_state_reused_without_refitting": True, "persistent_file_count": len(FILE_COVERAGE),
    "freshness_weight_reloaded": saved_recipe["freshness_weight"],
    "max_abs_blended_slate_difference": score_difference,
    "max_abs_catalogue_score_difference": catalogue_difference,
    "catalogue_rankings_match": ranking_matches,
    "passed": bool(len(difference) and np.isfinite(difference).all()
                   and difference.max() <= RELOAD_TOLERANCE and score_difference <= RELOAD_TOLERANCE
                   and catalogue_difference <= RELOAD_TOLERANCE and ranking_matches)}
D.write_json(RELOAD_CHECK, RUN_DIR / "reload_check.json")
D._copy_atomic(RUN_DIR / "reload_check.json", PERSIST_RUN_DIR / "reload_check.json")
assert RELOAD_CHECK["passed"], RELOAD_CHECK
log("Numerical reload check:", RELOAD_CHECK)

Numerical reload check: {'source': '/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/2.1.3_improved_run/20261104_175920_improved_seed42/model', 'requests': 8, 'candidate_rows': 96, 'max_abs_logit_difference': 0.0, 'tolerance': 0.0001, 'fitted_state_reused_without_refitting': True, 'persistent_file_count': 44, 'freshness_weight_reloaded': 2.0, 'max_abs_blended_slate_difference': 0.0, 'max_abs_catalogue_score_difference': 0.0, 'catalogue_rankings_match': True, 'passed': True}


## 9. Business interpretation and evaluator handoff

### Completed run and selected score

Run `20261104_175920_improved_seed42` fitted one LSTM on all 232,887 training
requests using the initial study's selected neural hyperparameters. The fixed
2,000-request validation sample jointly selected **epoch 6** and a **freshness
weight of 2**. Training stopped after epoch 8 without further Recall@100
improvement. The 72 completed Optuna records are nine freshness weights evaluated
at each of eight epochs, not 72 separate neural model fits.

The deployed ranking score for this experiment is
`neural_logit - 2 * log1p(article_age_hours)`. On the selection sample it achieved
Recall@100 of 0.8909 and observed-click nDCG@100 of 0.2746.

### Full-validation findings

All three comparisons below use the **same 29,110 validation requests and full
time-eligible catalogue**. The raw-neural row uses the selected model's weights
with the explicit freshness penalty removed; it is not a separately tuned fit.

| Ranking method | Macro known-click Recall@100 | Observed-click nDCG@100 |
| --- | ---: | ---: |
| Selected LSTM + freshness | **0.8970** | **0.2752** |
| Newest-first | 0.8902 | 0.2365 |
| Same LSTM, raw neural score | 0.6711 | 0.1855 |

The selected score recovered an average **89.70% of each request's known clicks**.
It exceeded newest-first by **0.68 percentage point** in Recall@100 and 0.0387
in nDCG@100. All 29,249 observed clicked articles were eligible, and no validation
request had an empty pool. The improvement therefore comes from ordering the
same eligible articles, not from changing catalogue coverage.

Freshness accounts for most of the retrieval performance. Adding the neural
score provides a modest further gain over newest-first, while the raw neural
score alone remains substantially weaker. Its 67.11% recall is also below the
initial run's 72.73%, so this experiment demonstrates improvement in the
**selected combined ranking score**, not an improvement in standalone neural
retrieval. Joint checkpoint selection targeted that combined score. These
results do not isolate an architectural benefit from attention or event order.

Validation was reused for model development and selection. No paired uncertainty
interval was produced for this final run, so the small gain over newest-first
is descriptive rather than a demonstrated statistically significant improvement.
Unknown outcomes for unshown articles remain unknown; these are observed-click
recovery measures rather than measured catalogue CTR, revenue or retention.

### Inference efficiency and completed exports

On the A100-SXM4-80GB, the controlled 32-request saved-model benchmark reproduced
the original logits exactly and matched the top 100 and top 200 at freshness
weights 0 and 2. Batched scoring was **1.60 times as fast**, and the complete
top-200 benchmark was **1.88 times as fast** as the original inference path on
the same GPU. These are small-batch benchmark results, not an end-to-end speed
claim against the earlier run on different hardware.

All **232,887 training, 29,110 validation and 29,142 holdout requests** received
their genuine top-200 exports. The inference progress logs approached completion
at 18.6, 2.6 and 2.3 minutes respectively; those readings exclude remaining
validation/copying work and are not total wall-clock run durations. Export checks
passed for all splits, including request coverage and eligible-pool counts.
Training predictions remain in-sample; holdout outcomes were not evaluated.

### Business use, persistence and handoff

This result supports keeping a single LSTM plus a fitted freshness term as the
Stage 1 relevance model. It balances recent news with learned reader/article
matching at a lower inference cost than a three-model ensemble. It does not
replace the separate business reranker. The first 100 articles form the working
shortlist, with the next 100 retained as a buffer; preserve the Stage 1 score and
rank when assessing the reranker's relevance and business trade-offs. The final
blended score is a ranking value, not a calibrated click probability.

The evaluator receives `rankings/validation/submission_top_200.parquet` for all
29,110 requests, retained-pair scores, complete recorded-slate diagnostics and
`run_config.json`. The persistent model bundle also stores the selected score
recipe, fitted preprocessing, meaningful feature/category maps, original article
IDs and all three helper modules. The reload check passed on eight requests and
96 recorded candidate pairs: raw logits, blended scores and retained catalogue
scores reproduced exactly, and catalogue rankings matched. The freshness weight
was recovered as 2 from the saved recipe.

Later evaluation can read the saved validation exports, while explainability can
load this complete scoring pipeline for selected cases without retraining or
repeating the full catalogue export. Explanations should distinguish learned
neural contributions from the explicit age penalty. Attention weights alone
are not causal explanations. Final holdout evaluation remains deferred until
the full pipeline, including business reranking, is frozen.


In [17]:
log("Handoff folder:", PERSIST_RUN_DIR)
log("Actual completed tuning trials:", TUNING_SUMMARY["completed_trials"],
    "| final epochs:", len(HISTORY), "| selected epoch:", BEST["epoch"])
log("All-request export counts:", {s: CATALOGUE_RECORDS[s]["integrity_check"] for s in D.SPLITS})
log("Selected freshness weight:", FRESHNESS_WEIGHT)
log("Full validation blended:", METRICS["full_validation_catalogue"])
log("Full validation raw neural:", RAW_NEURAL_CATALOGUE_METRICS)
log("Full validation newest-first:", NEWEST_FIRST_METRICS)
log("Inference benchmark:", BENCHMARK)
log("Persistent numerical reload passed:", RELOAD_CHECK["passed"])

Handoff folder: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/2.1.3_improved_run/20261104_175920_improved_seed42
Actual completed tuning trials: 72 | final epochs: 8 | selected epoch: 6
All-request export counts: {'train': {'requests': 232887, 'eligible_pairs': 28552935815, 'score_rows': 46577400, 'empty_pools': 0, 'all_request_ids_and_pool_counts_checked': True, 'score_ranking_requests_checked': 128}, 'validation': {'requests': 29110, 'eligible_pairs': 3608174696, 'score_rows': 5822000, 'empty_pools': 0, 'all_request_ids_and_pool_counts_checked': True, 'score_ranking_requests_checked': 128}, 'test': {'requests': 29142, 'eligible_pairs': 566598922, 'score_rows': 5828400, 'empty_pools': 0, 'all_request_ids_and_pool_counts_checked': True, 'score_ranking_requests_checked': 128}}
Selected freshness weight: 2.0
Full validation blended: {'requests_with_known_clicks_evaluated': 29110, 'requests_without_known_clicks_scored': 0, 'known_clicks_evaluated':